In [1]:
# Explainable machine learning for household electricity demand forecasting.
#
# Dataset  : UCI Individual Household Electric Power Consumption
#            (December 2006 to November 2010, one-minute resolution)
# Target   : Global_active_power (kW)
# Pipeline : XGBoost, LightGBM, CatBoost and Random Forest base learners
#            combined by a constrained convex blend. The Holt-Winters
#            decomposition is retained as an analysis layer only.


In [2]:
# Package installation

import subprocess, sys

_PACKAGES = [
    'pandas>=1.5', 'numpy>=1.23', 'matplotlib>=3.6',
    'seaborn>=0.12', 'scikit-learn>=1.3', 'statsmodels>=0.14',
    'shap>=0.42', 'lime>=0.2', 'scipy>=1.10',
    'xgboost>=1.7', 'lightgbm>=4.0', 'catboost>=1.2', 'joblib>=1.2',
    'arch>=5.3',  # circular block bootstrap for autocorrelated residuals
]
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q'] + _PACKAGES
)
print('Package installation complete.')


Package installation complete.


In [3]:
# Imports and global configuration

from __future__ import annotations
import gc, warnings, json, time
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, wilcoxon

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
)
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

from xgboost import XGBRegressor
import lightgbm
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.base import clone
import shap
from lime.lime_tabular import LimeTabularExplainer
import joblib


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Output directory
OUTPUT_DIR = Path('results')
OUTPUT_DIR.mkdir(exist_ok=True)

# Plot style
warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi':      110,
    'figure.figsize':  (12, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size':       10,
})
PALETTE = {
    'primary':   '#1f4e8c',
    'secondary': '#c0392b',
    'accent':    '#27ae60',
    'neutral':   '#7f8c8d',
}

# Figure conventions
plt.rcParams.update({
    'font.size':       11,
    'axes.titlesize':  12,
    'axes.labelsize':  11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'savefig.dpi':     400,
    'pdf.fonttype':    42, 
    'ps.fonttype':     42,
})

def savefig_pub(stem, fig=None, **kwargs):
    """Save a figure as a 400-dpi raster and a vector copy."""
    fig = fig if fig is not None else plt.gcf()
    kwargs.setdefault('bbox_inches', 'tight')
    fig.savefig(OUTPUT_DIR / f'{stem}.png', dpi=300, **kwargs)
    fig.savefig(OUTPUT_DIR / f'{stem}.pdf', **kwargs)

# Descriptive feature labels.


_RAW_VAR_LABELS = {
    'Global_active_power':   'Active power (kW)',
    'Global_reactive_power': 'Reactive power (kVAr)',
    'Voltage':               'Voltage (V)',
    'Global_intensity':      'Current intensity (A)',
    'Sub_metering_1':        'Kitchen sub-meter (Wh)',
    'Sub_metering_2':        'Laundry sub-meter (Wh)',
    'Sub_metering_3':        'Water-heater sub-meter (Wh)',
}

def _dur_label(m):
    """Readable duration for a horizon of m minutes."""
    m = int(m)
    if m % 1440 == 0:
        d = m // 1440
        return '1 day' if d == 1 else f'{d} days'
    if m % 60 == 0:
        h = m // 60
        return '1 hour' if h == 1 else f'{h} hours'
    return '1 minute' if m == 1 else f'{m} minutes'

def pub_label(feat):
    """Descriptive label for an engineered-feature name."""
    import re as _re
    f = str(feat)
    _fixed = {
        'diff_lag1_lag2':        'Load change (1 minute)',
        'diff_lag1_lag60':       'Load change (1 hour)',
        'gap_vs_yesterday':      'Change from previous day',
        'gap_vs_last_week':      'Change from previous week',
        'gap_vs_daily_mean':     'Deviation from daily mean',
        'gap_vs_weekly_mean':    'Deviation from weekly mean',
        'interaction_hour_lag1': 'Recent load by hour of day',
        'hour_sin':   'Hour of day (sine)',
        'hour_cos':   'Hour of day (cosine)',
        'day_sin':    'Day of week (sine)',
        'day_cos':    'Day of week (cosine)',
        'month_sin':  'Month of year (sine)',
        'month_cos':  'Month of year (cosine)',
        'mon_sin':    'Month of year (sine)',
        'mon_cos':    'Month of year (cosine)',
        'is_weekend': 'Weekend indicator',
        'hw_trend':        'Holt-Winters trend',
        'hw_seasonal':     'Holt-Winters seasonal term',
        'hw_residual':     'Holt-Winters residual',
        'hw_log_forecast': 'Holt-Winters forecast',
    }
    if f in _fixed:
        return _fixed[f]
    if f in _RAW_VAR_LABELS:
        return _RAW_VAR_LABELS[f]
    if f.endswith('_was_missing'):
        base = f[:-len('_was_missing')]
        base_lbl = _RAW_VAR_LABELS.get(base, base.replace('_', ' '))
        return f'Imputed reading ({base_lbl.split(" (")[0].lower()})'
    m = _re.search(r'_lag_(\d+)$', f)
    if m:
        return f'Load {_dur_label(int(m.group(1)))} earlier'
    m = _re.search(r'_roll_(mean|std|min|max)_(\d+)$', f)
    if m:
        stat = {'mean': 'Mean load', 'std': 'Standard deviation',
                'min': 'Minimum load', 'max': 'Maximum load'}[m.group(1)]
        return f'{stat} ({_dur_label(int(m.group(2)))})'
    m = _re.search(r'_ewm_(\d+)$', f)
    if m:
        return f'Smoothed load ({_dur_label(int(m.group(1)))})'
    m = _re.search(r'^f(?:ourier_)?([dw])_(sin|cos)_(\d+)$', f)
    if m:
        scope = 'Daily' if m.group(1) == 'd' else 'Weekly'
        trig = 'sine' if m.group(2) == 'sin' else 'cosine'
        return f'{scope} harmonic {m.group(3)} ({trig})'
    m = _re.search(r'hw_res(?:idual)?_lag_?(\d+)$', f)
    if m:
        return f'Holt-Winters residual ({_dur_label(int(m.group(1)))} earlier)'
    return f.replace('_', ' ')

print(f"Environment configured. Random seed: {RANDOM_STATE}")


Environment configured. Random seed: 42


In [4]:
# Data acquisition


import urllib.request, zipfile

DATA_URL  = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip'
DATA_ZIP  = Path('household_power_consumption.zip')
DATA_FILE = Path('household_power_consumption.txt')

if not DATA_FILE.exists():
    print('Downloading dataset...')
    urllib.request.urlretrieve(DATA_URL, DATA_ZIP)
    with zipfile.ZipFile(DATA_ZIP, 'r') as z:
        z.extractall('.')
    print('Download complete.')
else:
    print('Dataset already present on disk.')


Dataset already present on disk.


In [5]:
# Parsing and initial data-quality assessment
# The character '?' denotes a missing reading. The Date and Time fields are
# combined into a regular one-minute DatetimeIndex.

df_raw = pd.read_csv(
    DATA_FILE, sep=';', na_values=['?'],
    low_memory=False, dtype=str
)
dt = pd.to_datetime(
    df_raw['Date'] + ' ' + df_raw['Time'],
    format='%d/%m/%Y %H:%M:%S', errors='coerce'
)
df_raw = df_raw.drop(columns=['Date', 'Time'])
df_raw.index = dt
df_raw = df_raw[~df_raw.index.isna()].sort_index()
df_raw = df_raw[~df_raw.index.duplicated(keep='first')]
df_raw = df_raw.asfreq('1min')  # inserting empty rows for any skipped minutes

NUMERIC_COLS = [
    'Global_active_power', 'Global_reactive_power', 'Voltage',
    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]
for col in NUMERIC_COLS:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

print(f"Shape          : {df_raw.shape}")
print(f"Date range     : {df_raw.index.min()} to {df_raw.index.max()}")
print(f"Missing values : {df_raw[NUMERIC_COLS].isna().sum().sum():,}")

summary = df_raw[NUMERIC_COLS].describe().T
summary['missing_pct'] = df_raw[NUMERIC_COLS].isna().mean() * 100
print("\n--- Descriptive statistics ---")
print(summary.round(4).to_string())


Shape          : (2075259, 7)
Date range     : 2006-12-16 17:24:00 to 2010-11-26 21:02:00
Missing values : 181,853

--- Descriptive statistics ---
                           count      mean     std      min      25%      50%      75%      max  missing_pct
Global_active_power    2049280.0    1.0916  1.0573    0.076    0.308    0.602    1.528   11.122       1.2518
Global_reactive_power  2049280.0    0.1237  0.1127    0.000    0.048    0.100    0.194    1.390       1.2518
Voltage                2049280.0  240.8399  3.2400  223.200  238.990  241.010  242.890  254.150       1.2518
Global_intensity       2049280.0    4.6278  4.4444    0.200    1.400    2.600    6.400   48.400       1.2518
Sub_metering_1         2049280.0    1.1219  6.1530    0.000    0.000    0.000    0.000   88.000       1.2518
Sub_metering_2         2049280.0    1.2985  5.8220    0.000    0.000    0.000    1.000   80.000       1.2518
Sub_metering_3         2049280.0    6.4584  8.4372    0.000    0.000    1.000   17.000   3